# 5-minute tour of `mergeplg`

This is a very brief overview of features available in the `mergeplg` package for merging rainfall data from multiple sensors (CMLs, PWSs, SMLs) with gridded data such as radar. A more comprehensive overview can be found in the `mergeplg` [documentation](https://mergeplg.readthedocs.io).

In [ ]:
import matplotlib.pyplot as plt

import poligrain as plg
import mergeplg as mrg

## Load example data

We use a subset of the OpenMRG dataset (Andersson et al., 2022) which contains radar, CML and rain gauge data in Gothenburg, Sweden. `poligrain` provides a convenient loader for this dataset.

In [ ]:
(
    ds_rad,
    ds_cmls,
    ds_gauges_municp,
    ds_gauge_smhi,
) = plg.example_data.load_openmrg(data_dir="example_data", subset="8d")

## Merge radar with CML data

`mergeplg` provides classes to merge gridded data (e.g. radar) with point or line observations (e.g. rain gauges or CMLs). Here we use `MergeDifferenceIDW` to adjust the radar field with CML-derived rain rates using inverse distance weighting of the difference field.

In [ ]:
# Set up the merging object which internally calculates
# and stores the CML-radar intersection weights
merge_diff_idw = mrg.merge.MergeDifferenceIDW(
    ds_rad=ds_rad.R,
    ds_cmls=ds_cmls,
    method='additive',
    nnear=12,
)

# Aggregate to 1h
R_radar = ds_rad.R.resample(time='1h').mean()
R_cmls = ds_cmls.isel(sublink_id=0).R.resample(time='1h').mean()
R_gauges = ds_gauges_municp.rainfall_amount.resample(time='1h').sum()

In [ ]:
# Do the adjustment for one selected timestamp
t = '2015-07-26 03:00:00'
R_grid_idw = merge_diff_idw(
    da_rad=R_radar.sel(time=t),
    da_cmls=R_cmls.sel(time=t),
)
R_grid_idw

In [ ]:
# Plot the original and the adjusted radar field
fig, axs = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
vmin, vmax = 0, 15

plg.plot_map.plot_plg(
    da_grid=R_radar.sel(time=t),
    da_cmls=R_cmls.sel(time=t),
    da_gauges=R_gauges.sel(time=t),
    vmin=vmin,
    vmax=vmax,
    ax=axs[0],
    kwargs_cmls_plot={"line_width": 1},
)
axs[0].set_title("Original radar")

plg.plot_map.plot_plg(
    da_grid=R_grid_idw,
    da_cmls=R_cmls.sel(time=t),
    da_gauges=R_gauges.sel(time=t),
    vmin=vmin,
    vmax=vmax,
    ax=axs[1],
    kwargs_cmls_plot={"line_width": 1},
)
axs[1].set_title("Radar adjusted with CMLs");

## Summary

That's the core merging workflow in `mergeplg`: load gridded and point/line data → set up a `MergeDifferenceIDW` object → adjust the gridded field with the observations. Explore the [documentation](https://mergeplg.readthedocs.io) for other merging methods and options.